In [ ]:
!pip install -q transformers datasets evaluate openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 9.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.8

In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from evaluate import load
import nltk
nltk.download("punkt")

import pandas as pd
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Load and preprocess data
df_raw = pd.read_excel(filename)
df_raw.columns = df_raw.columns.str.strip()
df_raw = df_raw.rename(columns={"Sentence #": "sentence", "word": "word", "aspect": "aspect", "categort": "category", "polarity": "polarity"})
df_raw = df_raw.fillna("O")
grouped = df_raw.groupby("sentence")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Saving All_dataa.xlsx to All_dataa.xlsx


/usr/local/lib/python3.11/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [ ]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=d84cba037a463d83570a8356322dc02b21b108cc78dacd904dd52fcd00851015
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval


In [ ]:
import pandas as pd
import numpy as np
import re
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    DataCollatorWithPadding
)
from evaluate import load

# --- 2. Load Model and Tokenizer ---
model_checkpoint = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# --- 3. Prepare the Raw Data ---
def normalize_text(text):
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()

# Clean columns and rename them
df_raw.columns = df_raw.columns.str.strip()
df_raw = df_raw.rename(columns={
    "Sentence #": "sentence",
    "word": "word",
    "aspect": "aspect",
    "categort": "category",
    "polarity": "polarity"
})

# Fill missing values and clean words
df_raw = df_raw.fillna("O")
df_raw["word"] = df_raw["word"].astype(str).apply(normalize_text)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

grouped = df_raw.groupby("sentence")
final_data = []

for _, group in grouped:
    words = group["word"].tolist()
    aspects = group["aspect"].tolist()

    final_tokens = []
    final_labels = []

    prev_aspect = "O"

    for word, aspect in zip(words, aspects):
        tokens = tokenizer.tokenize(word)

        if aspect != "O":
            bio = "B" if prev_aspect == "O" else "I"
        else:
            bio = "O"

        for idx, token in enumerate(tokens):
            final_tokens.append(token)
            if bio == "O":
                final_labels.append("O")
            else:
                final_labels.append(bio if idx == 0 else "I")

        prev_aspect = aspect

    final_data.append({"tokens": final_tokens, "tags": final_labels})

df = pd.DataFrame(final_data)

# label maps
tag_list = ["O", "B", "I"]
tag2id = {tag: i for i, tag in enumerate(tag_list)}
id2tag = {i: tag for tag, i in tag2id.items()}

def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            current_word = word_id
            label = -100 if word_id is None else tag2id[labels[word_id]]
            new_labels.append(label)
        elif word_id is None:
            new_labels.append(-100)
        else:
            label = labels[word_id]
            new_labels.append(tag2id[label])
    return new_labels

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, padding=True)
    all_labels = examples["tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))
    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

# Dataset
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

# ModernBERT
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(tag2id),
    id2label=id2tag,
    label2id=tag2id,
)

#  Data Collator(padding)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

metric = load("seqeval")


def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_predictions = [
        [id2tag[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2tag[l] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }
args = TrainingArguments(
    output_dir="modernbert-absa-t1",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=10,
    per_device_eval_batch_size=10,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True
)
#  Train and Evaluate
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
trainer.train()

trainer.evaluate()

Map:   0%|          | 0/3876 [00:00<?, ? examples/s]

Map:   0%|          | 0/970 [00:00<?, ? examples/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.367793,0.516960,0.555331,0.535459,0.860553
2,0.436300,0.348613,0.581427,0.582996,0.582210,0.871420
3,0.296900,0.343479,0.581219,0.595142,0.588098,0.872876


{'eval_loss': 0.3434789478778839,
 'eval_precision': 0.5812191103789127,
 'eval_recall': 0.5951417004048583,
 'eval_f1': 0.5880980163360559,
 'eval_accuracy': 0.8728760384509199,
 'eval_runtime': 2.7705,
 'eval_samples_per_second': 350.12,
 'eval_steps_per_second': 35.012,
 'epoch': 3.0}

In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import evaluate

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load and filter data
# Use "sentence" instead of "Sentence #"
df = df_raw[df_raw["category"] != "O"][["sentence", "word", "category"]]
# Encode labels
label_list = sorted(df["category"].unique())
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}
df["label"] = df["category"].map(label2id)

# Group words into sentences
sentences = []
labels = []

# Use "sentence" instead of "Sentence #"
for sent_id, group in df.groupby("sentence"):
    words = group["word"].tolist()
    label_ids = group["label"].tolist()
    sentences.append(words)
    labels.append(label_ids)

# Create Dataset
dataset = Dataset.from_dict({"tokens": sentences, "labels": labels})

# Train/test split
dataset = dataset.train_test_split(test_size=0.2, seed=42)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

# Tokenization + align labels
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True,
    )

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # special tokens ignored in loss
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])  # subword, same label
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenize dataset
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

# Load model
model = AutoModelForTokenClassification.from_pretrained(
    "answerdotai/ModernBERT-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

# Data collator
data_collator = DataCollatorForTokenClassification(tokenizer)

# Load metric
metric = evaluate.load("accuracy")

# Define compute_metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_predictions = [
        [pred for pred, label in zip(prediction, label_row) if label != -100]
        for prediction, label_row in zip(predictions, labels)
    ]
    true_labels = [
        [label for label in label_row if label != -100]
        for label_row in labels
    ]

    return metric.compute(predictions=[p for preds in true_predictions for p in preds],
                           references=[l for labels in true_labels for l in labels])

# Training arguments
training_args = TrainingArguments(
    output_dir="./modernbert_category_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,  # Use the correct compute_metrics function
)

# Train model
trainer.train()

# Evaluate model
final_results = trainer.evaluate()
print(f" Final Evaluation Results:\n{final_results}")

Map:   0%|          | 0/3869 [00:00<?, ? examples/s]

Map:   0%|          | 0/968 [00:00<?, ? examples/s]

Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-8-fa305d0ff20b>:129: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.419458,0.852848
2,No log,0.329371,0.885464
3,0.480000,0.339680,0.890286


 Final Evaluation Results:
{'eval_loss': 0.3396804630756378, 'eval_accuracy': 0.8902859844008508, 'eval_runtime': 3.4097, 'eval_samples_per_second': 283.898, 'eval_steps_per_second': 17.89, 'epoch': 3.0}


In [ ]:
import pandas as pd
import numpy as np
import re

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import evaluate

# --- Step 1: Prepare Polarity Labels ---
def extract_polarity_data(df):
    # Check if 'Sentence #' is in the columns, if not use 'sentence'
    sentence_col = 'Sentence #' if 'Sentence #' in df.columns else 'sentence'
    df = df.query("polarity != 'O' and word.str.len() >= 2 and word.str.strip() != ''", engine='python')
    return df[[sentence_col, 'word', 'polarity']]

df_polarity = extract_polarity_data(df_raw)

# --- Step 2: Create Label Mappings ---
label_list = sorted(df_polarity["polarity"].unique())
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

# --- Step 3: Prepare the dataset ---
# Group by Sentence to rebuild sentences
sentences = []
labels = []

# Check if 'Sentence #' is in the columns, if not use 'sentence'
sentence_col = 'Sentence #' if 'Sentence #' in df_polarity.columns else 'sentence'

for sent_id, group in df_polarity.groupby(sentence_col):
    words = group["word"].tolist()
    polarities = group["polarity"].tolist()
    sentences.append(words)
    labels.append([label2id[label] for label in polarities])

# Create a HuggingFace Dataset
dataset = Dataset.from_dict({"tokens": sentences, "labels": labels})

# Split into train/test
dataset = dataset.train_test_split(test_size=0.2, seed=42)

# --- Step 4: Tokenizer ---
model_checkpoint = "aubmindlab/bert-base-arabertv02"  # or any ModernBERT you use
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True,
    )

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # Mask tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                # Same word as previous subword
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

# --- Step 5: Load Model ---
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# --- Step 6: Training arguments ---
args = TrainingArguments(
    output_dir="modernbert-absa-tokens",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
)

# --- Step 7: Data collator for token classification ---
data_collator = DataCollatorForTokenClassification(tokenizer)

# --- Step 8: Metrics ---
metric = evaluate.load("accuracy")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [pred for pred, label in zip(prediction, label_list) if label != -100]
        for prediction, label_list in zip(predictions, labels)
    ]
    true_labels = [
        [label for label in label_list if label != -100]
        for label_list in labels
    ]

    return metric.compute(predictions=[p for preds in true_predictions for p in preds],
                           references=[l for labels in true_labels for l in labels])

# --- Step 9: Trainer ---
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# --- Step 10: Train and Evaluate ---
trainer.train()
trainer.evaluate()

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<ipython-input-5-e94afdce5df6>:131: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.301491,0.878528
2,No log,0.306572,0.890602
3,0.281000,0.300168,0.889002


{'eval_loss': 0.3065715432167053,
 'eval_accuracy': 0.8906022694210067,
 'eval_runtime': 2.113,
 'eval_samples_per_second': 458.106,
 'eval_steps_per_second': 28.868,
 'epoch': 3.0}

In [ ]:
# Save the best model

trainer.save_model("modernbert-absa-t1")
tokenizer.save_pretrained("modernbert-absa-t1")

('modernbert-absa-t1/tokenizer_config.json',
 'modernbert-absa-t1/special_tokens_map.json',
 'modernbert-absa-t1/tokenizer.json')

In [ ]:
trainer.save_model("./modernbert_category_model")
tokenizer.save_pretrained("./modernbert_category_model")

('./modernbert_category_model/tokenizer_config.json',
 './modernbert_category_model/special_tokens_map.json',
 './modernbert_category_model/tokenizer.json')

In [ ]:
trainer.save_model("modernbert_polarity_model")
tokenizer.save_pretrained("modernbert_polarity_model")

('modernbert_polarity_model/tokenizer_config.json',
 'modernbert_polarity_model/special_tokens_map.json',
 'modernbert_polarity_model/tokenizer.json')

In [ ]:
import random, json, torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification

# Load ModernBERT models and tokenizer
model_t1 = AutoModelForTokenClassification.from_pretrained("answerdotai/ModernBERT-base").eval()
model_t3 = AutoModelForSequenceClassification.from_pretrained("answerdotai/ModernBERT-base").eval()
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

# Define label mappings for aspect classification
id2label = {
    0: "O",  # Outside aspect
    1: "B",  # Beginning of aspect
    2: "I"   # Inside aspect
}
label2id = {v: k for k, v in id2label.items()}  # Reverse mapping if needed

# Define label mappings for sentiment classification
id2pol = {
    0: "neutral",  # Neutral sentiment
    1: "positive", # Positive sentiment
    2: "negative"  # Negative sentiment
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_t1 = model_t1.to(device)
model_t3 = model_t3.to(device)

# Extract sentences from df_raw
grouped = df_raw.groupby("sentence")
sentences = [(" ".join(group["word"].astype(str)), name) for name, group in grouped]

# Select 5 random sentences
random_sentences = random.sample(sentences, 5)

results = []

# Process each sentence
for original_sentence, _ in random_sentences:
    clean_text = original_sentence  # Clean text without preprocessing
    clean_text = clean_text.encode("utf-8").decode("utf-8")  # Ensure correct encoding

    encoding = tokenizer(clean_text, return_tensors="pt", return_offsets_mapping=True, truncation=True, max_length=128)

    # Get word_ids before moving to device
    word_ids = encoding.word_ids()
    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])

    # Move to device
    inputs = {k: v.to(device) for k, v in encoding.items() if k in ["input_ids", "attention_mask"]}

    # Get predictions from T1 model (for aspect classification)
    with torch.no_grad():
        logits = model_t1(**inputs).logits
    predictions = torch.argmax(logits, dim=-1)[0].tolist()

    word_tokens, word_labels = {}, {}
    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        token = tokens[idx]
        label = id2label.get(predictions[idx], "O")  # Default to "O" if not in labels
        if word_id not in word_tokens:
            word_tokens[word_id] = []
            word_labels[word_id] = label
        word_tokens[word_id].append(token)

    words_info = []
    for word_id in sorted(word_tokens.keys()):
        token_group = word_tokens[word_id]
        word_text = tokenizer.convert_tokens_to_string(token_group).strip()
        bio_label = word_labels[word_id]
        sentiment = "-"

        if bio_label in ["B", "I"]:  # If the word is part of an aspect
            inp = tokenizer(word_text, return_tensors="pt", truncation=True, padding="max_length", max_length=32)
            inp = {k: v.to(device) for k, v in inp.items()}
            with torch.no_grad():
                out = model_t3(**inp)  # Sentiment model inference
            sentiment = id2pol[torch.argmax(out.logits, dim=1).item()]

        words_info.append({
            "word": word_text,
            "tokens": token_group,
            "BIO": bio_label,
            "sentiment": sentiment
        })

    results.append({
        "sentence": original_sentence,
        "words": words_info
    })

# Print the results in JSON format
print(json.dumps(results, ensure_ascii=False, indent=2))


Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[
  {
    "sentence": "يعني اطلب الساعه واتواصل معاكم الساعه تقولون لي لم يتم قبول الطلب والمشكله ماخذين الفلوس ومكتوب دقيقه اذا في ساعه ماقبلتو الطلب يعني الطلب بتجيبونه عقب ساعات هذا بعد ماتقبلونه الصراحه خدمتكم جدا سيئه وهذا مش اول موقف لي معاكم",
    "words": [
      {
        "word": "يعني",
        "tokens": [
          "ÙĬ",
          "Ø¹",
          "ÙĨ",
          "ÙĬ"
        ],
        "BIO": "B",
        "sentiment": "neutral"
      },
      {
        "word": "اطلب",
        "tokens": [
          "ĠØ§Ø",
          "·",
          "ÙĦ",
          "Ø¨"
        ],
        "BIO": "B",
        "sentiment": "neutral"
      },
      {
        "word": "الساعه",
        "tokens": [
          "ĠØ§ÙĦ",
          "Ø³",
          "Ø§Ø",
          "¹",
          "Ùĩ"
        ],
        "BIO": "B",
        "sentiment": "neutral"
      },
      {
        "word": "واتواصل",
        "tokens": [
          "ĠÙĪ",
          "Ø§Øª",
          "ÙĪ",
          "Ø§Ø",
          "µ",
          "ÙĦ"
 